## Parte 2. Revision Del Dataset

In [1]:
import pandas as pd
df = pd.read_csv("ventas_ecommerce_limpio.csv")

In [2]:
## 1. Muestra las primeras filas.
df.head()

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta
0,1001,2026-07-01,Ana Lopez,Mouse,Accesorios,1,250.0,Efectivo,Cuernavaca,250.0
1,1002,2026-07-01,Luis Perez,Teclado,Accesorios,1,650.0,Tarjeta,Jiutepec,650.0
2,1003,2026-07-02,Sofia Ruiz,Audifonos,Accesorios,1,900.0,Tarjeta,Temixco,900.0
3,1004,2026-07-02,Pedro Mata,Webcam,Accesorios,1,800.0,Efectivo,Cuernavaca,800.0
4,1005,2026-07-03,Laura Diaz,Cable HDMI,Accesorios,2,180.0,Efectivo,Jiutepec,360.0


In [4]:
## 2. Revisa las columnas.

df.columns

Index(['id_venta', 'fecha', 'cliente', 'producto', 'categoria', 'cantidad',
       'precio_unitario', 'metodo_pago', 'ciudad', 'total_venta'],
      dtype='object')

In [7]:
## 3. Muestra cuantas filas y columnas tiene.
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 60
Columnas: 10


In [8]:
## 4. Revisa si hay valores nulos.
df.isnull().sum()

id_venta           0
fecha              0
cliente            0
producto           0
categoria          0
cantidad           0
precio_unitario    0
metodo_pago        0
ciudad             0
total_venta        0
dtype: int64

In [9]:
## 5. Verifica que exista la columna `total_venta`.
"total_venta" in df.columns

True

In [10]:
## 6. Verifica que `total_venta` coincida con `cantidad * precio_unitario`.
df["total_calculado"] = df["cantidad"] * df["precio_unitario"]
df["coincide_total"] = df["total_venta"] == df["total_calculado"]
df["coincide_total"].value_counts()

coincide_total
True    60
Name: count, dtype: int64

* 7. Escribe una observacion breve sobre el estado del dataset: Hay 60 filas, 10 columnas, sin nulos, total_venta sí existe y coincide al 100% con cantidad * precio_unitario

## Parte 3. Variable Objetivo

In [17]:
## Crear la columna y contar cuantas ventas quedaron como 1 y cuantas como 0:
df["venta_alta"] = df["total_venta"].apply(lambda x: 1 if x >= 1000 else 0)
df.head()

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta,total_calculado,coincide_total,venta_alta
0,1001,2026-07-01,Ana Lopez,Mouse,Accesorios,1,250.0,Efectivo,Cuernavaca,250.0,250.0,True,0
1,1002,2026-07-01,Luis Perez,Teclado,Accesorios,1,650.0,Tarjeta,Jiutepec,650.0,650.0,True,0
2,1003,2026-07-02,Sofia Ruiz,Audifonos,Accesorios,1,900.0,Tarjeta,Temixco,900.0,900.0,True,0
3,1004,2026-07-02,Pedro Mata,Webcam,Accesorios,1,800.0,Efectivo,Cuernavaca,800.0,800.0,True,0
4,1005,2026-07-03,Laura Diaz,Cable HDMI,Accesorios,2,180.0,Efectivo,Jiutepec,360.0,360.0,True,0


In [18]:
df["venta_alta"].value_counts()

venta_alta
1    40
0    20
Name: count, dtype: int64

* Por que venta_alta es la variable objetivo? Porque es la columna que queremos que el modelo aprenda a predecir

## Parte 4. Variables De Entrada

In [22]:
## Vaurbles de entrada y objetivo:
X = df[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]
y = df["venta_alta"]

In [25]:
## Ahora hacer: Convierte variables categoricas con `pd.get_dummies()`; Guarda la lista de columnas generadas; 
X = pd.get_dummies(X)
columnas_examen_modelo = X.columns.tolist()
columnas_examen_modelo

['cantidad',
 'precio_unitario',
 'categoria_Accesorios',
 'categoria_Electronica',
 'categoria_Muebles',
 'metodo_pago_Efectivo',
 'metodo_pago_Tarjeta',
 'metodo_pago_Transferencia',
 'ciudad_Cuernavaca',
 'ciudad_Emiliano Zapata',
 'ciudad_Jiutepec',
 'ciudad_Temixco']

In [26]:
## Muestra las primeras filas de `X` despues de `get_dummies()`.
X.head()

,cantidad,precio_unitario,categoria_Accesorios,categoria_Electronica,categoria_Muebles,metodo_pago_Efectivo,metodo_pago_Tarjeta,metodo_pago_Transferencia,ciudad_Cuernavaca,ciudad_Emiliano Zapata,ciudad_Jiutepec,ciudad_Temixco
0,1,250.0,True,False,False,True,False,False,True,False,False,False
1,1,650.0,True,False,False,False,True,False,False,False,True,False
2,1,900.0,True,False,False,False,True,False,False,False,False,True
3,1,800.0,True,False,False,True,False,False,True,False,False,False
4,2,180.0,True,False,False,True,False,False,False,False,True,False


* Por que no se debe usar total_venta como variable de entrada si venta_alta se creo a partir de total_venta? Porque sería fuga de información

## Parte 5. Entrenamiento Y Evaluacion

In [29]:
## Hacer:
## 1. Divide los datos en entrenamiento y prueba.
## 2. Usa 80% entrenamiento y 20% prueba.
## 3. Usa `random_state=42`.
## 4. Entrena el modelo.
## 5. Genera predicciones con los datos de prueba.
## 6. Calcula exactitud.
## 7. Muestra matriz de confusion.
## 8. Crea una tabla llamada `resultados_prueba` con: text, valor_real, prediccion, coincide
## 9. Cuenta cuantos aciertos y cuantos errores hubo.

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [31]:
modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [32]:
y_pred = modelo.predict(X_test)
y_pred

array([0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1])

In [33]:
exactitud = accuracy_score(y_test, y_pred)
print("Exactitud:", exactitud)

Exactitud: 1.0


In [34]:
matriz = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Matriz de confusión")
print(matriz)

Matriz de confusión
[[4 0]
 [0 8]]


In [35]:
resultados_prueba = pd.DataFrame({"valor_real": y_test.values, "prediccion": y_pred})
resultados_prueba["coincide"] = resultados_prueba["valor_real"] == resultados_prueba["prediccion"]
resultados_prueba

,valor_real,prediccion,coincide
0,0,0,True
1,0,0,True
2,1,1,True
3,1,1,True
4,0,0,True
5,1,1,True
6,1,1,True
7,1,1,True
8,0,0,True
9,1,1,True


In [36]:
resultados_prueba["coincide"].value_counts()

coincide
True    12
Name: count, dtype: int64

In [37]:
aciertos = resultados_prueba[resultados_prueba["coincide"] == True]
errores = resultados_prueba[resultados_prueba["coincide"] == False]
print("Aciertos:", len(aciertos))
print("Errores:", len(errores))

Aciertos: 12
Errores: 0


# Preguntas:
1. Cual fue la exactitud? 1.0 (100%).
2. Cuantos aciertos tuvo el modelo? 12 aciertos.
3. Cuantos errores tuvo el modelo? 0 errores.
4. Que indica la matriz de confusion? Muestra aciertos y errores por clase: [[4,0],[0,8]] — 4 "no alta" y 8 "alta" bien clasificadas, sin errores.
5. Una buena exactitud significa que el modelo ya es perfecto? Explica. No. Aquí dio 1.0, pero con datos nuevos (categoría/ciudad desconocidas) el modelo sí falló después.

## Parte 6. Guardar Modelo Y Columnas

In [38]:
## Hacer:
## 1. Usa `joblib`.
## 2. Guarda el modelo entrenado.
## 3. Guarda la lista de columnas usadas durante el entrenamiento.
## 4. Verifica que los archivos aparezcan en tu carpeta.

In [39]:
import joblib
import os

joblib.dump(modelo, "modelo_examen_venta_alta.pkl")
joblib.dump(columnas_examen_modelo, "columnas_examen_modelo.pkl")

['columnas_examen_modelo.pkl']

In [40]:
print("Modelo guardado:", os.path.exists("modelo_examen_venta_alta.pkl"))
print("Columnas guardadas:", os.path.exists("columnas_examen_modelo.pkl"))

Modelo guardado: True
Columnas guardadas: True


# Preguntas:
1. Para que sirve guardar el modelo? Para poder reutilizarlo después sin necesidad de volver a entrenarlo cada vez.
2. Para que sirve guardar las columnas del entrenamiento? Para saber exactamente qué columnas espera el modelo al recibir datos nuevos, y poder alinearlas con reindex()
3. Que problema puede aparecer si no guardas las columnas? Al usar get_dummies() en datos nuevos podrían generarse columnas distintas a las del entrenamiento, y sin las columnas originales guardadas el modelo fallaria

## Parte 7. Ventas Nuevas

In [43]:
## Crear un archivo que tenga:
## 1. Incluye al menos 4 ventas claramente altas.
## 2. Incluye al menos 4 ventas claramente no altas.
## 3. Incluye al menos 2 ventas cercanas al limite de 1000.
## 4. Incluye al menos 1 categoria nueva.
## 5. Incluye al menos 1 ciudad nueva.
## 6. No copies exactamente las ventas del pre examen.
## 7. No uses las mismas ventas que otro companero.

In [44]:
examen_ventas_nuevas = pd.DataFrame([
    # Ventas claramente altas (4)
    {"id_venta": 5001, "fecha": "2026-08-01", "cliente": "Orlando Ruiz", "producto": "Laptop",
     "categoria": "Electronica", "cantidad": 1, "precio_unitario": 15000.0, "metodo_pago": "Tarjeta", "ciudad": "Cuernavaca"},
    {"id_venta": 5002, "fecha": "2026-08-02", "cliente": "Sebas Ruiz", "producto": "Monitor",
     "categoria": "Electronica", "cantidad": 1, "precio_unitario": 6000.0, "metodo_pago": "Transferencia", "ciudad": "Jiutepec"},
    {"id_venta": 5003, "fecha": "2026-08-03", "cliente": "Jose Ruiz", "producto": "Television",
     "categoria": "Electronica", "cantidad": 1, "precio_unitario": 9500.0, "metodo_pago": "Tarjeta", "ciudad": "Temixco"},
    {"id_venta": 5004, "fecha": "2026-08-04", "cliente": "Astrid Ruiz", "producto": "Refrigerador",
     "categoria": "Electrodomesticos", "cantidad": 1, "precio_unitario": 8000.0, "metodo_pago": "Efectivo", "ciudad": "Emiliano Zapata"},
    # Ventas claramente no altas (4)
    {"id_venta": 5005, "fecha": "2026-08-05", "cliente": "Cesar Ruiz", "producto": "Mouse",
     "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 200.0, "metodo_pago": "Efectivo", "ciudad": "Cuernavaca"},
    {"id_venta": 5006, "fecha": "2026-08-06", "cliente": "Andre Ruiz", "producto": "Cable USB",
     "categoria": "Accesorios", "cantidad": 2, "precio_unitario": 100.0, "metodo_pago": "Tarjeta", "ciudad": "Jiutepec"},
    {"id_venta": 5007, "fecha": "2026-08-07", "cliente": "Erick Ruiz", "producto": "Audifonos",
     "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 350.0, "metodo_pago": "Efectivo", "ciudad": "Temixco"},
    {"id_venta": 5008, "fecha": "2026-08-08", "cliente": "Nicole Ruiz", "producto": "Mouse Pad",
     "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 150.0, "metodo_pago": "Tarjeta", "ciudad": "Emiliano Zapata"},
    # Ventas cercanas al limite de 1000 (2)
    {"id_venta": 5009, "fecha": "2026-08-09", "cliente": "Evelyn Ruiz", "producto": "Bocina Bluetooth",
     "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 950.0, "metodo_pago": "Transferencia", "ciudad": "Cuernavaca"},
    {"id_venta": 5010, "fecha": "2026-08-10", "cliente": "Julio Ruiz", "producto": "Teclado",
     "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 1050.0, "metodo_pago": "Efectivo", "ciudad": "Cuautla"},
])
examen_ventas_nuevas

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad
0,5001,2026-08-01,Orlando Ruiz,Laptop,Electronica,1,15000.0,Tarjeta,Cuernavaca
1,5002,2026-08-02,Sebas Ruiz,Monitor,Electronica,1,6000.0,Transferencia,Jiutepec
2,5003,2026-08-03,Jose Ruiz,Television,Electronica,1,9500.0,Tarjeta,Temixco
3,5004,2026-08-04,Astrid Ruiz,Refrigerador,Electrodomesticos,1,8000.0,Efectivo,Emiliano Zapata
4,5005,2026-08-05,Cesar Ruiz,Mouse,Accesorios,1,200.0,Efectivo,Cuernavaca
5,5006,2026-08-06,Andre Ruiz,Cable USB,Accesorios,2,100.0,Tarjeta,Jiutepec
6,5007,2026-08-07,Erick Ruiz,Audifonos,Accesorios,1,350.0,Efectivo,Temixco
7,5008,2026-08-08,Nicole Ruiz,Mouse Pad,Accesorios,1,150.0,Tarjeta,Emiliano Zapata
8,5009,2026-08-09,Evelyn Ruiz,Bocina Bluetooth,Accesorios,1,950.0,Transferencia,Cuernavaca
9,5010,2026-08-10,Julio Ruiz,Teclado,Accesorios,1,1050.0,Efectivo,Cuautla


In [45]:
examen_ventas_nuevas.to_csv("examen_ventas_nuevas.csv", index=False)
print("Archivo guardado: examen_ventas_nuevas.csv")

Archivo guardado: examen_ventas_nuevas.csv


## Parte 8. Cargar Modelo Y Predecir

In [57]:
## Hacer:
## 1. Carga `modelo_examen_venta_alta.pkl`.
## 2. Carga `columnas_examen_modelo.pkl`.
## 3. Carga `examen_ventas_nuevas.csv`.
## 4. Selecciona las mismas variables de entrada usadas en entrenamiento.
## 5. Aplica `pd.get_dummies()`.
## 6. Alinea columnas con `reindex()`.
## 7. Genera predicciones.
## 8. Agrega la columna: prediccion_venta_alta
## 9. Agrega la columna: interpretacion_prediccion
## 10. Guarda el resultado como: examen_predicciones.csv

In [58]:
modelo_cargado = joblib.load("modelo_examen_venta_alta.pkl")
columnas_modelo_cargadas = joblib.load("columnas_examen_modelo.pkl")

In [59]:
examen_ventas_nuevas = pd.read_csv("examen_ventas_nuevas.csv")
examen_ventas_nuevas

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad
0,5001,2026-08-01,Orlando Ruiz,Laptop,Electronica,1,15000.0,Tarjeta,Cuernavaca
1,5002,2026-08-02,Sebas Ruiz,Monitor,Electronica,1,6000.0,Transferencia,Jiutepec
2,5003,2026-08-03,Jose Ruiz,Television,Electronica,1,9500.0,Tarjeta,Temixco
3,5004,2026-08-04,Astrid Ruiz,Refrigerador,Electrodomesticos,1,8000.0,Efectivo,Emiliano Zapata
4,5005,2026-08-05,Cesar Ruiz,Mouse,Accesorios,1,200.0,Efectivo,Cuernavaca
5,5006,2026-08-06,Andre Ruiz,Cable USB,Accesorios,2,100.0,Tarjeta,Jiutepec
6,5007,2026-08-07,Erick Ruiz,Audifonos,Accesorios,1,350.0,Efectivo,Temixco
7,5008,2026-08-08,Nicole Ruiz,Mouse Pad,Accesorios,1,150.0,Tarjeta,Emiliano Zapata
8,5009,2026-08-09,Evelyn Ruiz,Bocina Bluetooth,Accesorios,1,950.0,Transferencia,Cuernavaca
9,5010,2026-08-10,Julio Ruiz,Teclado,Accesorios,1,1050.0,Efectivo,Cuautla


In [60]:
X_nuevas = examen_ventas_nuevas[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]
X_nuevas = pd.get_dummies(X_nuevas)
X_nuevas = X_nuevas.reindex(columns=columnas_modelo_cargadas, fill_value=0)
X_nuevas

,cantidad,precio_unitario,categoria_Accesorios,categoria_Electronica,categoria_Muebles,metodo_pago_Efectivo,metodo_pago_Tarjeta,metodo_pago_Transferencia,ciudad_Cuernavaca,ciudad_Emiliano Zapata,ciudad_Jiutepec,ciudad_Temixco
0,1,15000.0,False,True,0,False,True,False,True,False,False,False
1,1,6000.0,False,True,0,False,False,True,False,False,True,False
2,1,9500.0,False,True,0,False,True,False,False,False,False,True
3,1,8000.0,False,False,0,True,False,False,False,True,False,False
4,1,200.0,True,False,0,True,False,False,True,False,False,False
5,2,100.0,True,False,0,False,True,False,False,False,True,False
6,1,350.0,True,False,0,True,False,False,False,False,False,True
7,1,150.0,True,False,0,False,True,False,False,True,False,False
8,1,950.0,True,False,0,False,False,True,True,False,False,False
9,1,1050.0,True,False,0,True,False,False,False,False,False,False


In [61]:
predicciones = modelo_cargado.predict(X_nuevas)

examen_ventas_nuevas["prediccion_venta_alta"] = predicciones

examen_ventas_nuevas["interpretacion_prediccion"] = examen_ventas_nuevas["prediccion_venta_alta"].map({
    0: "Venta no alta",
    1: "Venta alta"
})

examen_ventas_nuevas

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion
0,5001,2026-08-01,Orlando Ruiz,Laptop,Electronica,1,15000.0,Tarjeta,Cuernavaca,1,Venta alta
1,5002,2026-08-02,Sebas Ruiz,Monitor,Electronica,1,6000.0,Transferencia,Jiutepec,1,Venta alta
2,5003,2026-08-03,Jose Ruiz,Television,Electronica,1,9500.0,Tarjeta,Temixco,1,Venta alta
3,5004,2026-08-04,Astrid Ruiz,Refrigerador,Electrodomesticos,1,8000.0,Efectivo,Emiliano Zapata,1,Venta alta
4,5005,2026-08-05,Cesar Ruiz,Mouse,Accesorios,1,200.0,Efectivo,Cuernavaca,0,Venta no alta
5,5006,2026-08-06,Andre Ruiz,Cable USB,Accesorios,2,100.0,Tarjeta,Jiutepec,0,Venta no alta
6,5007,2026-08-07,Erick Ruiz,Audifonos,Accesorios,1,350.0,Efectivo,Temixco,0,Venta no alta
7,5008,2026-08-08,Nicole Ruiz,Mouse Pad,Accesorios,1,150.0,Tarjeta,Emiliano Zapata,0,Venta no alta
8,5009,2026-08-09,Evelyn Ruiz,Bocina Bluetooth,Accesorios,1,950.0,Transferencia,Cuernavaca,0,Venta no alta
9,5010,2026-08-10,Julio Ruiz,Teclado,Accesorios,1,1050.0,Efectivo,Cuautla,0,Venta no alta


In [62]:
examen_ventas_nuevas.to_csv("examen_predicciones.csv", index=False)
print("Archivo guardado: examen_predicciones.csv")

Archivo guardado: examen_predicciones.csv


## Pregunta:
1. Que podria pasar si no usas reindex antes de predecir? Las columnas generadas por get_dummies() en los datos nuevos probablemente no coincidirían en cantidad ni en orden con las columnas que el modelo espera, por lo que daria error.

## Parte 9. Auditoria Del Modelo

In [66]:
## Hacr:
## 1. Calcula: total_estimado = cantidad * precio_unitario
## 2. Crea: venta_alta_real_estimada
## 3. Compara: prediccion_venta_alta y venta_alta_real_estimada
## 4. Crea la columna: coincide
## 5. Cuenta cuantas predicciones coincidieron y cuantas no.
## 6. Filtra las ventas que no coincidieron.
## 7. Revisa si los errores estan cerca del limite de 1000.
## 8. Guarda de nuevo `examen_predicciones.csv` con las columnas de auditoria.

In [67]:
examen_ventas_nuevas["total_estimado"] = examen_ventas_nuevas["cantidad"] * examen_ventas_nuevas["precio_unitario"]
examen_ventas_nuevas.head()

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide
0,5001,2026-08-01,Orlando Ruiz,Laptop,Electronica,1,15000.0,Tarjeta,Cuernavaca,1,Venta alta,15000.0,1,True
1,5002,2026-08-02,Sebas Ruiz,Monitor,Electronica,1,6000.0,Transferencia,Jiutepec,1,Venta alta,6000.0,1,True
2,5003,2026-08-03,Jose Ruiz,Television,Electronica,1,9500.0,Tarjeta,Temixco,1,Venta alta,9500.0,1,True
3,5004,2026-08-04,Astrid Ruiz,Refrigerador,Electrodomesticos,1,8000.0,Efectivo,Emiliano Zapata,1,Venta alta,8000.0,1,True
4,5005,2026-08-05,Cesar Ruiz,Mouse,Accesorios,1,200.0,Efectivo,Cuernavaca,0,Venta no alta,200.0,0,True


In [68]:
examen_ventas_nuevas["venta_alta_real_estimada"] = examen_ventas_nuevas["total_estimado"].apply(
    lambda x: 1 if x >= 1000 else 0
)
examen_ventas_nuevas.head()

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide
0,5001,2026-08-01,Orlando Ruiz,Laptop,Electronica,1,15000.0,Tarjeta,Cuernavaca,1,Venta alta,15000.0,1,True
1,5002,2026-08-02,Sebas Ruiz,Monitor,Electronica,1,6000.0,Transferencia,Jiutepec,1,Venta alta,6000.0,1,True
2,5003,2026-08-03,Jose Ruiz,Television,Electronica,1,9500.0,Tarjeta,Temixco,1,Venta alta,9500.0,1,True
3,5004,2026-08-04,Astrid Ruiz,Refrigerador,Electrodomesticos,1,8000.0,Efectivo,Emiliano Zapata,1,Venta alta,8000.0,1,True
4,5005,2026-08-05,Cesar Ruiz,Mouse,Accesorios,1,200.0,Efectivo,Cuernavaca,0,Venta no alta,200.0,0,True


In [69]:
examen_ventas_nuevas["coincide"] = examen_ventas_nuevas["prediccion_venta_alta"] == examen_ventas_nuevas["venta_alta_real_estimada"]

examen_ventas_nuevas[[
    "id_venta", "total_estimado", "prediccion_venta_alta", "venta_alta_real_estimada", "coincide"
]]

,id_venta,total_estimado,prediccion_venta_alta,venta_alta_real_estimada,coincide
0,5001,15000.0,1,1,True
1,5002,6000.0,1,1,True
2,5003,9500.0,1,1,True
3,5004,8000.0,1,1,True
4,5005,200.0,0,0,True
5,5006,200.0,0,0,True
6,5007,350.0,0,0,True
7,5008,150.0,0,0,True
8,5009,950.0,0,0,True
9,5010,1050.0,0,1,False


In [70]:
examen_ventas_nuevas["coincide"].value_counts()

coincide
True     9
False    1
Name: count, dtype: int64

In [71]:
errores = examen_ventas_nuevas[examen_ventas_nuevas["coincide"] == False]
errores

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide
9,5010,2026-08-10,Julio Ruiz,Teclado,Accesorios,1,1050.0,Efectivo,Cuautla,0,Venta no alta,1050.0,1,False


In [72]:
examen_ventas_nuevas["distancia_a_1000"] = (examen_ventas_nuevas["total_estimado"] - 1000).abs()
cerca_limite = examen_ventas_nuevas[examen_ventas_nuevas["distancia_a_1000"] <= 200]
cerca_limite

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide,distancia_a_1000
8,5009,2026-08-09,Evelyn Ruiz,Bocina Bluetooth,Accesorios,1,950.0,Transferencia,Cuernavaca,0,Venta no alta,950.0,0,True,50.0
9,5010,2026-08-10,Julio Ruiz,Teclado,Accesorios,1,1050.0,Efectivo,Cuautla,0,Venta no alta,1050.0,1,False,50.0


In [74]:
examen_ventas_nuevas.to_csv("examen_predicciones.csv", index=False)
print("Archivo actualizado: examen_predicciones.csv")

Archivo actualizado: examen_predicciones.csv


## Preguntas:
1. Cuantas ventas nuevas evaluaste? 10 ventas nuevas.
2. Cuantas fueron predichas como venta alta? 4 ventas (5001, 5002, 5003, 5004).
3. Cuantas fueron predichas como venta no alta? 6 ventas (5005, 5006, 5007, 5008, 5009 y 5010).
4. Cuantas coincidieron con la regla manual? 9 de las 10 predicciones coincidieron.
5. Cuantas no coincidieron? 1 predicción no coincidió.
6. Que ventas no coincidieron? La venta 5010
7. Los errores estuvieron cerca del limite de 1000? Si, el único error está dentro del rango de ±200 del límite de 1000.
8. Que paso con la categoria nueva? La categoría "Electrodomesticos"no existía en el entrenamiento. Al aplicar reindex(), esa columna se descartó y la fila quedó con todas las columnas de categoría en 0.
9. Que paso con la ciudad nueva? La ciudad "Cuautla" no está en columnas_examen_modelo, así que tras reindex() esa fila quedó con todas las columnas de ciudad en 0.

## Conclusion:
* Nombre: Orlando Ruiz Santos
* Grupo: 9° A
* Materia: Extracción de conocimiento de Bases de Datos
* Exactitud obtenida: 1.0 (100%)
* Ventas nuevas evaluadas: 10
* Coincidencias: 9
* Errores: 1
* Conclusion breve: El modelo funcionó bien en las pruebas iniciales, pero al usarlo con ventas nuevas se equivocó en 1 de 10, justo en una venta con una ciudad que no conocía y un monto cercano al límite. Esto me hizo ver que aunque el modelo tenga buena exactitud, no siempre acierta con datos que nunca ha visto.